# COCO ingestion with Daft and Apache Paimon

This notebook mirrors the PySpark pipeline but executes the image ingestion and inference workflow with [Daft](https://github.com/Eventual-Inc/Daft).
It downloads the COCO 2017 training images, stores them in a blob-backed Paimon table, uses Daft to drive PyTorch inference, and persists the resulting feature blobs to a derived table.

## Setup
The environment needs Daft, the Hugging Face `datasets` loader, and the PyTorch + torchvision stack.  This cell installs the dependencies directly in the kernel.

In [1]:
import sys
!{sys.executable} -m pip install --quiet --upgrade pip
!{sys.executable} -m pip install --quiet "daft[torch]" datasets pillow torchvision torch --extra-index-url https://download.pytorch.org/whl/cpu

## Imports and catalog bootstrap
We rely on the lightweight helper module under `paimon-daft/` to bridge Paimon with Daft.

In [2]:
import datetime as dt
import io
import os
from pathlib import Path
from typing import Dict, List

import daft
import datasets
from PIL import Image
import torch
from torchvision import models, transforms

import pyarrow as pa

import sys
sys.path.append(str(Path.cwd().parent / "paimon-daft"))
sys.path.append(str(Path.cwd().parent / "paimon-python"))

from paimon_daft import (
    ensure_catalog,
    ensure_database,
    ensure_table,
    read_table_as_daft,
    write_daft_to_table,
)
from pypaimon.schema.data_types import AtomicType, DataField

WAREHOUSE_ROOT = Path.cwd().parent / "data" / "paimon_daft_warehouse"
WAREHOUSE_ROOT.mkdir(parents=True, exist_ok=True)

CATALOG_CONF = {"warehouse": f"file://{WAREHOUSE_ROOT}"}
CATALOG = ensure_catalog(CATALOG_CONF)
ensure_database(CATALOG, "coco_demo")
RAW_TABLE = "coco_demo.raw_images"
FEATURE_TABLE = "coco_demo.feature_vectors"
TODAY = dt.date.today().isoformat()
TARGET_IMAGE_COUNT = 100_000
DEMO_IMAGE_CAP = int(os.environ.get("PAIMON_DAFT_IMAGE_CAP", "32"))

## Create blob-backed tables
The raw table keeps the JPEG payload in a `BLOB` column with the required table properties.  The feature table will store the extracted embeddings as blobs as well.

In [3]:
raw_fields = [
    DataField(0, "capture_date", AtomicType("STRING")),
    DataField(1, "image_id", AtomicType("BIGINT")),
    DataField(2, "source", AtomicType("STRING")),
    DataField(3, "width", AtomicType("INT")),
    DataField(4, "height", AtomicType("INT")),
    DataField(5, "content", AtomicType("BLOB")),
]

feature_fields = [
    DataField(0, "capture_date", AtomicType("STRING")),
    DataField(1, "image_id", AtomicType("BIGINT")),
    DataField(2, "label", AtomicType("STRING")),
    DataField(3, "score", AtomicType("DOUBLE")),
    DataField(4, "feature_blob", AtomicType("BLOB")),
]

ensure_table(
    CATALOG,
    RAW_TABLE,
    fields=raw_fields,
    partition_keys=["capture_date"],
    options={
        "blob-field": "content",
        "blob-as-descriptor": "true",
        "data-evolution.enabled": "true",
        "row-tracking.enabled": "true",
    },
    overwrite=True,
)

ensure_table(
    CATALOG,
    FEATURE_TABLE,
    fields=feature_fields,
    partition_keys=["capture_date"],
    options={
        "blob-field": "feature_blob",
        "blob-as-descriptor": "true",
        "data-evolution.enabled": "true",
        "row-tracking.enabled": "true",
    },
    overwrite=True,
)

## Download COCO images
We stream the COCO 2017 training set via 🤗 Datasets.  The demo cap keeps the runtime manageable for validation environments while the production parameter covers the 100k+ target.

In [4]:
def iter_coco_samples(limit: int):
    dataset = datasets.load_dataset(
        "coco_captions",
        "2017",
        split="train",
        streaming=True,
    )
    for idx, sample in enumerate(dataset):
        if idx >= limit:
            break
        pil_image = sample["image"].convert("RGB")
        yield sample["image_id"], pil_image

coco_samples = list(iter_coco_samples(min(TARGET_IMAGE_COUNT, DEMO_IMAGE_CAP)))
len(coco_samples)

DatasetNotFoundError: Dataset 'coco_captions' doesn't exist on the Hub or cannot be accessed.

## Ingest images into the raw Paimon table
The helper converts the Python list into a Daft dataframe and tries to commit it via the Paimon batch writer.

In [5]:
def build_raw_records(samples):
    records: List[Dict] = []
    for image_id, pil_image in samples:
        buffer = io.BytesIO()
        pil_image.save(buffer, format="JPEG")
        payload = buffer.getvalue()
        records.append(
            {
                "capture_date": TODAY,
                "image_id": int(image_id),
                "source": "coco2017/train",
                "width": pil_image.width,
                "height": pil_image.height,
                "content": payload,
            }
        )
    return records

raw_records = build_raw_records(coco_samples)
raw_schema = pa.schema([
    ("capture_date", pa.string()),
    ("image_id", pa.int64()),
    ("source", pa.string()),
    ("width", pa.int32()),
    ("height", pa.int32()),
    ("content", pa.large_binary()),
])

raw_arrow = pa.Table.from_pylist(raw_records, schema=raw_schema)
raw_df = daft.from_arrow(raw_arrow)
write_daft_to_table(CATALOG, RAW_TABLE, raw_df)
raw_df

NameError: name 'coco_samples' is not defined

## Attempt Daft-based feature extraction
This section mirrors the PySpark workflow: it loads the raw table through Daft, applies a PyTorch detector, and tries to persist the embeddings.

In [6]:
detection_model = models.detection.fasterrcnn_resnet50_fpn(weights=models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
detection_model.eval()
preprocess = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ]
)

PERSON_OR_ANIMAL = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18}

def extract_features(row):
    image_bytes = row["content"]
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    tensor = preprocess(image)
    with torch.no_grad():
        outputs = detection_model([tensor])
    boxes = outputs[0]["boxes"]
    labels = outputs[0]["labels"].tolist()
    scores = outputs[0]["scores"].tolist()
    hits = [
        (label, score)
        for label, score in zip(labels, scores)
        if label in PERSON_OR_ANIMAL and score >= 0.4
    ]
    if not hits:
        return None
    best_label, best_score = hits[0]
    embedding = tensor.flatten().numpy().tobytes()
    return {
        "capture_date": row["capture_date"],
        "image_id": row["image_id"],
        "label": str(best_label),
        "score": float(best_score),
        "feature_blob": embedding,
    }

raw_table_df = read_table_as_daft(CATALOG, RAW_TABLE)
feature_rows = (
    raw_table_df
    .to_pydict()
)
len(feature_rows["image_id"])

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


  0% 0.00/160M [00:00<?, ?B/s]

  1% 1.00M/160M [00:00<00:16, 10.1MB/s]

  2% 3.62M/160M [00:00<00:08, 18.5MB/s]

  4% 6.12M/160M [00:00<00:07, 20.9MB/s]

...

 96% 153M/160M [00:06<00:00, 22.9MB/s]

 98% 156M/160M [00:07<00:00, 22.8MB/s]

 99% 158M/160M [00:07<00:00, 23.0MB/s]

100% 160M/160M [00:07<00:00, 23.2MB/s]

0

In [7]:
def build_feature_records(raw_dict):
    output: List[Dict] = []
    for idx in range(len(raw_dict["image_id"])):
        row = {key: raw_dict[key][idx] for key in raw_dict}
        maybe_record = extract_features(row)
        if maybe_record is not None:
            output.append(maybe_record)
    return output

feature_records = build_feature_records(raw_table_df.to_pydict())
feature_arrow = pa.Table.from_pylist(
    feature_records,
    schema=pa.schema([
        ("capture_date", pa.string()),
        ("image_id", pa.int64()),
        ("label", pa.string()),
        ("score", pa.float64()),
        ("feature_blob", pa.large_binary()),
    ]),
)
feature_df = daft.from_arrow(feature_arrow)
write_daft_to_table(CATALOG, FEATURE_TABLE, feature_df)
feature_df

capture_dateString,image_idInt64,labelString,scoreFloat64,feature_blobBinary


## Display sample detections
If the Daft workflow succeeds this cell renders ten detections.

In [8]:
feature_sample = read_table_as_daft(CATALOG, FEATURE_TABLE)
feature_sample.head(10)

AttributeError: 'DataFrame' object has no attribute 'head'